
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [23]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="gpt-oss:20b")

In [24]:
response = model.invoke("Please provide details about the movie 'Inception'.")
print(response.content)

## Inception (2010) – Overview

| Item | Details |
|------|---------|
| **Director** | Christopher Nolan |
| **Screenwriter** | Christopher Nolan |
| **Production Company** | Syncopy Inc. |
| **Distributed by** | Warner Bros. Pictures |
| **Runtime** | 148 min |
| **Genre** | Science‑fiction action‑thriller |
| **Language** | English |
| **Release Dates** | 16 July 2010 (United States) – 30 July 2010 (UK) |
| **Budget** | ~$160 million |
| **Box‑office** | $836 million worldwide |
| **MPAA Rating** | PG‑13 |

---

### 1. Plot (Brief)

*Dom Cobb* (Leonardo DiCaprio) is a “extractor” who enters others’ dreams to steal secrets. He is offered a seemingly impossible task: *plant* an idea (an “inception”) into the mind of Robert Fischer (Cillian Murphy), the heir to a business empire. Cobb assembles a team—Arthur (Joseph Gordon-Levitt), Ariadne (Elliot Page), Eames (Tom Hardy), and Yusuf (Dileep Rao).  

They design a multi‑layered dream “architecture,” where each layer is a deeper subconsci

In [25]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The rating of the movie")    

In [26]:

model_with_structured_output = model.with_structured_output(Movie, include_raw=True)

response = model_with_structured_output.invoke(
    "Please provide details about the movie 'Inception'."
)
response

{'raw': AIMessage(content='{"title":"Inception","year":2010,"director":"Christopher\u202fNolan","rating":8.8}', additional_kwargs={}, response_metadata={'model': 'gpt-oss:20b', 'created_at': '2026-11-25T12:29:22.077969Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3907188542, 'load_duration': 2684125, 'prompt_eval_count': 77, 'prompt_eval_duration': 72450000, 'eval_count': 190, 'eval_duration': 3813927000, 'logprobs': None, 'model_name': 'gpt-oss:20b', 'model_provider': 'ollama'}, id='lc_run--01a212ae-89d9-72e9-ba2f-afc13b819616-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 190, 'total_tokens': 267}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher\u202fNolan', rating=8.8),
 'parsing_error': None}

### Nested Structure

In [29]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)

title='Inception' year=2010 cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal')] genres=['Action', 'Adventure', 'Sci‑Fi', 'Thriller'] budget=200000000.0


### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [32]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict = model.with_structured_output(MovieDict)
response = model_withtypedict.invoke("Please provide the details of the movie 'Inception'")
response

{'title': 'Inception',
 'year': 2010,
 'director': 'Christopher Nolan',
 'rating': 8.8}

In [33]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie 'Inception'")
response

{'title': 'Inception',
 'year': 2010,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb / Arthur'},
  {'name': 'Joseph Gordon‑Levitt', 'role': 'Arthur'},
  {'name': 'Ellen Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Eames'},
  {'name': 'Ken Watanabe', 'role': 'Saito'},
  {'name': 'Cillian Murphy', 'role': 'Robert Fischer'},
  {'name': 'Marion Cotillard', 'role': 'Mal'}],
 'genres': ['Action', 'Adventure', 'Sci‑Fi', 'Thriller'],
 'budget': 160000000.0}

In [35]:
model.profile

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")
# can be integrated with a provider strategy to extract contact info from textSS
agent = create_agent(
    model=model,  # Use the ChatOllama model
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result)
print('*' * 100)
print(result['structured_response'])

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='56ca7659-11ac-4ae5-8aa8-05bdb01e4181'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:20b', 'created_at': '2026-11-25T13:20:51.238276Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1161443708, 'load_duration': 1788292, 'prompt_eval_count': 176, 'prompt_eval_duration': 58935000, 'eval_count': 55, 'eval_duration': 1077193000, 'logprobs': None, 'model_name': 'gpt-oss:20b', 'model_provider': 'ollama'}, id='lc_run--01a212dd-b799-7165-9240-f4ac3bd2d927-0', tool_calls=[{'name': 'ContactInfo', 'args': {'email': 'john@example.com', 'name': 'John Doe', 'phone': '(555) 123-4567'}, 'id': 'c61cd1d3-4d97-40f9-9fe5-fd8235cc3146', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 176, 'output_tokens': 55, 'total_tokens': 231}), ToolMessage(content="Returning structured respon

In [40]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model=model,  # Use the ChatOllama model
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [41]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model=model,  # Use the ChatOllama model
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')